<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 4</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Log Segments, Indexes</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Understand how Kafka stores data on disk.</p>
</div>

---

## 🎯 Overview

Kafka is a distributed commit log. Under the hood, partitions are stored as directories on the broker's disk, and data is broken into **Segment Files**.

Key concepts you will explore:
- 📁 **.log files:** The actual message data.
- 🔍 **.index / .timeindex files:** Mappings to quickly find messages by offset or timestamp.

---

## ⚙️ Prerequisites

We will use the standard single-node cluster from the root directory for this lab.

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Ensure any previous multi-broker clusters are stopped. Run <code>docker-compose down</code> in Lab3 before proceeding.
</div>

---

## <span style="color: #11998e;">Step 1:</span> Start the Kafka Cluster


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d

---

## <span style="color: #11998e;">Step 2:</span> Log Segments and Indexes

Let's create a topic named `segment-demo` and intentionally set the **segment size very small** (1MB) so we can see Kafka roll over to new files quickly. (The default is 1GB).

In [ ]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic segment-demo \
  --partitions 1 \
  --replication-factor 1 \
  --config segment.bytes=1048576

Now, let's use the Kafka Producer Performance Test tool to generate a lot of data quickly and fill up those 1MB segments.

In [ ]:
!docker exec kafka kafka-producer-perf-test \
  --topic segment-demo \
  --num-records 50000 \
  --record-size 1024 \
  --throughput -1 \
  --producer-props bootstrap.servers=localhost:9092

---

## <span style="color: #11998e;">Step 3:</span> Inspect the Filesystem

Let's peek directly into the broker's disk where the data is stored (`/var/lib/kafka/data`). We will look inside the `segment-demo-0` directory.

Notice the **.log**, **.index**, and **.timeindex** files.

In [ ]:
!docker exec kafka bash -c "ls -lh /var/lib/kafka/data/segment-demo-0"

**💡 File Naming Convention:**
The filenames (`00000000000000000000.log`, etc.) represent the **Base Offset** of that segment. For example, if a file is named `00000000000000045612.log`, it means the very first message stored in that file is offset `45,612`. This allows Kafka to instantly know which file contains a requested offset without opening it!

---
Let's look at the actual `.log` file! We can use `--print-data-log` to read the binary file and show the physical contents (including the payload of the messages).

In [ ]:
# Note: Replace the filename below if it doesn't match your output above!
!docker exec kafka kafka-dump-log \
  --print-data-log \
  --files /var/lib/kafka/data/segment-demo-0/00000000000000000000.log

You can actually use a Kafka tool to read the `.index` files to see how Kafka maps an Offset to a Physical Position in the `.log` file.

*(If the index file name below does not exist, look at the output from the previous cell and replace the filename!)*

In [ ]:
!docker exec kafka kafka-dump-log \
  --files /var/lib/kafka/data/segment-demo-0/00000000000000000000.index

---

## 🧹 Clean Up

Stop the cluster and remove volumes to clean up the disk space.

In [ ]:
!docker-compose -f ../../docker-compose.yml down -v

<div style="background-color: rgba(17, 153, 142, 0.1); border: 1px solid rgba(17, 153, 142, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #11998e; margin-bottom: 10px;">🎉 Lab 4 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've peered under the hood at Kafka's storage layer.</p>
</div>